# 실습 9 (심화) · AI SOC 미니 프로젝트

지금까지의 실습을 하나의 파이프라인으로 묶습니다.

| 단계 | 내용 | 실습 |
|---|---|---|
| 0 | 로그 읽기 + 인젝션 시험 입력 1줄 추가 | 4 · 8 |
| 1 | 가명 마스킹 · 재검사 | 2b |
| 2 | 인젝션 의심 줄은 LLM에 보내지 않고 사람 검토로 분리 | 8-1 |
| 3 | 나머지 줄 LLM 판정 (DoS · Exploits · Reconnaissance · Normal) | 3 · 4 |
| 4 | 정답과 비교, 보고서 저장 (soc_report.json) | 4 |

In [1]:
from openai import OpenAI

ENV = "mono"          # "ollama" = 내 PC,  "mono" = MonoRouter(원격)
if ENV == "mono":
    client = OpenAI(base_url="https://monogpt.kr/api/monorouter/v1/openai/v1",
                    api_key="key")
    MODEL = "gpt-5.5"
else:
    client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    MODEL = "llama3.2:3b"
print(ENV, MODEL)

mono gpt-5.5


In [2]:
def ask(prompt, system=None, temperature=0.2, show=True):
    """질문을 보내고 답을 받아 돌려줌. show=True면 답이 만들어지는 대로 화면에 표시"""
    messages = [{"role": "user", "content": prompt}]
    if system:
        messages.insert(0, {"role": "system", "content": system})
    limit = {"max_tokens": 500} if ENV == "ollama" else {}   # 작은 모델이 끝없이 길게 답하는 것을 막음
    stream = client.chat.completions.create(model=MODEL, messages=messages,
                                            temperature=temperature, stream=True, **limit)
    answer = ""
    for chunk in stream:
        piece = chunk.choices[0].delta.content if chunk.choices else None
        if piece:
            answer += piece
            if show:
                print(piece, end="", flush=True)
    if show:
        print()
    return answer

In [3]:
import json, re

def parse_json(raw):
    """답에서 { ... } 부분만 꺼내 JSON으로 읽음. 없거나 깨졌으면 None"""
    m = re.search(r"\{.*\}", raw or "", re.DOTALL)
    try:
        return json.loads(m.group(0)) if m else None
    except json.JSONDecodeError:
        return None

## 0 · 1) 로그 읽기와 마스킹

In [4]:
import re
import pandas as pd

LINES = open("sample_logs.txt", encoding="utf-8").read().splitlines()
LINES += [l for l in open("injection_test.txt", encoding="utf-8").read().splitlines() if l and not l.startswith("#")]

INTERNAL_IP = r"\b149\.171\.126\.\d{1,3}\b"
table = {}
for line in LINES:
    for ip in re.findall(INTERNAL_IP, line):
        table.setdefault(ip, f"SRV_{len(table) + 1:03d}")
masked = [re.sub(INTERNAL_IP, lambda m: table[m.group(0)], l) for l in LINES]
assert not any(re.search(INTERNAL_IP, l) for l in masked), "마스킹 누락 → 전송 중단"
print(f"{len(masked)}줄 (마지막 줄 = 인젝션 시험 입력), 내부 서버 {len(table)}개 가명 처리")

13줄 (마지막 줄 = 인젝션 시험 입력), 내부 서버 8개 가명 처리


## 2 · 3) 인젝션 검사 → LLM 판정

In [5]:
KEYWORDS = ["ignore previous", "ignore all", "system prompt", "you are now", "jailbreak",
            "이전 지시 무시", "시스템 프롬프트", "이제부터 너는"]
TYPES = ["DoS", "Exploits", "Reconnaissance", "Normal"]
PROMPT = """한 줄 = 네트워크 연결 1건 (UNSW-NB15). state=연결 상태(FIN 정상 종료, INT 응답 없이 끝남), spkts/dpkts=송신/수신 패킷,
sbytes/dbytes=송신/수신 바이트. SRV_001 같은 값은 가명 처리한 우리 서버.
공격 유형은 DoS · Exploits · Reconnaissance · Normal 중 하나만 고른다.
로그: {log}
JSON으로만 답한다. {{"attack_type": "4개 중 하나", "risk_score": 0~100 정수, "reason": "한 줄 설명"}}"""

report = []
for i, line in enumerate(masked, 1):
    print(f"[{i}/{len(masked)}]", end=" ", flush=True)
    hit = next((k for k in KEYWORDS if k in line.lower()), None)
    if hit:                                                 # LLM이 지시문을 따를 수 있어 보내지 않음
        r = {"attack_type": "판정불가", "reason": f"인젝션 의심 키워드 '{hit}' → 사람 검토"}
    else:
        r = None
        for _ in range(2):
            d = parse_json(ask(PROMPT.format(log=line[28:]), system="JSON만 출력하는 보안 분석가.", show=False))
            if d and d.get("attack_type") in TYPES:
                r = d
                break
        r = r or {"attack_type": "판정불가", "reason": "형식 오류 → 사람 검토"}
    print(r["attack_type"], "|", str(r.get("reason"))[:70])
    report.append(dict(r, line=i, masked_log=line))

[1/13] Exploits | 공격 대역으로 보이는 외부 출발지에서 내부 HTTP 서버로 정상 종료된 단일 세션이며 DoS/Recon 패턴은 약하지만 웹 서
[2/13] Normal | HTTP 연결이 FIN으로 정상 종료되었고 송수신 패킷/바이트와 접속 빈도가 낮아 정상 트래픽으로 판단됨
[3/13] Normal | DNS 요청/응답이 송수신 2패킷씩 정상적으로 완료되었고 트래픽량과 연결 빈도가 낮아 공격 징후가 약합니다.
[4/13] Normal | UDP 통신이 짧고 송수신 패킷/바이트가 균형적이며 상태가 CON이고 반복 횟수 지표도 낮아 정상 연결로 판단됨
[5/13] Normal | FIN으로 정상 종료되었고 송수신 패킷/바이트가 균형적이며 연결 빈도 지표도 낮아 정상 FTP-data 트래픽으로 판단
[6/13] Normal | SMTP 연결이 FIN으로 정상 종료되었고 양방향 패킷/바이트 교환이 자연스러우며 DoS·Exploits·Reconnaissa
[7/13] Normal | SSH 세션이 FIN으로 정상 종료되었고 송수신 패킷/바이트가 균형적이며 반복 접속 지표도 낮아 정상 트래픽으로 판단됨
[8/13] Reconnaissance | UDP 111 포트로 매우 짧은 무응답(INT) 탐지성 트래픽이며 패킷/바이트가 적어 서비스 스캔 가능성이 높음
[9/13] Exploits | 외부 공격 대역으로 보이는 출발지에서 서버의 TCP 111/rpcbind 포트로 정상 세션이 성립되어 소량의 데이터 교환이 발
[10/13] Exploits | 공격 대역으로 알려진 출발지에서 HTTP 세션이 정상 종료(FIN)되었지만 양방향 소량 페이로드 교환이 있어 웹 취약점 시도 
[11/13] Exploits | 알려진 UNSW-NB15 공격 대역 출발지에서 내부 HTTP 서버로 정상 종료된 소량 TCP 세션으로, DoS나 Recon 징
[12/13] Exploits | 공격 IP 대역으로 알려진 외부 호스트가 POP3 서비스에 정상 종료(FIN)된 세션을 만들었

## 4) 정답과 비교, 보고서 저장

In [6]:
import json

truth = pd.read_csv("../data/sample_logs_truth.csv").set_index("line")["attack_cat"]
t = pd.DataFrame(report)
t["정답"] = t["line"].map(truth).fillna("인젝션 시험 입력")
t["일치"] = t["attack_type"] == t["정답"]
print(t[["line", "attack_type", "정답", "일치"]].to_string(index=False))
print(f"\n위협 판정 {t['attack_type'].isin(TYPES[:3]).sum()}건, 판정불가(사람 검토) {(t['attack_type'] == '판정불가').sum()}건, "
      f"정답 일치 {t['일치'].sum()}/{len(truth)}")

json.dump(report, open("soc_report.json", "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print("저장: soc_report.json (가명 로그만 들어 있음)")

 line    attack_type             정답    일치
    1       Exploits            DoS False
    2         Normal         Normal  True
    3         Normal         Normal  True
    4         Normal         Normal  True
    5         Normal         Normal  True
    6         Normal         Normal  True
    7         Normal         Normal  True
    8 Reconnaissance Reconnaissance  True
    9       Exploits Reconnaissance False
   10       Exploits       Exploits  True
   11       Exploits            DoS False
   12       Exploits       Exploits  True
   13           판정불가      인젝션 시험 입력 False

위협 판정 6건, 판정불가(사람 검토) 1건, 정답 일치 9/12
저장: soc_report.json (가명 로그만 들어 있음)


### 발표할 것
- 정답과 다른 줄과 그 이유, 판정불가로 분리된 줄과 처리 방법
- 인젝션 시험 입력이 LLM에 가지 않고 사람 검토로 분리되었는지
- 같은 출발지(175.45.176.1)의 정찰 → 침투 → DoS 흐름을 한 줄 분석으로 잡았는지, 개선 방안